# Train a DDPM from Scratch on MNIST

Companion notebook for the [Diffusion Models](https://codemia.ai/courses/deep_learning_foundations/diffusion_models) lesson on Codemia.

**What you will build**: a full Denoising Diffusion Probabilistic Model on MNIST — linear noise schedule, tiny UNet with sinusoidal timestep conditioning, the closed-form forward process, and an intermediate-timestep sampling visualization that lets you *watch* random noise condense into a digit.

**Time**: ~75 minutes (roughly 50 of those are training).

**GPU required**: yes. The model is tiny, but a 200-step reverse process over a few hundred test samples is brutally slow on CPU. Free-tier Colab T4 is plenty.

## Setup

In [ ]:
%pip install -q torch torchvision matplotlib

In [ ]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')
if device == 'cuda':
    print('  GPU:', torch.cuda.get_device_name(0))
else:
    print('  ! Sampling will be very slow without a GPU. Reduce N_TEST_SAMPLES below.')

## Concept recap

DDPM defines a **forward process** that gradually corrupts a clean image $x_0$ into pure Gaussian noise $x_T$ over $T$ steps. The corruption is fixed (no parameters to learn) and is engineered so that for *any* timestep $t$ you can sample $x_t$ in closed form:
$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
where $\bar\alpha_t$ is the cumulative product of $1 - \beta_t$ along the noise schedule. This shortcut is what makes training tractable — every batch, we pick a random $t$ per example, jump there in one shot, and never iterate the chain.

Training is then almost embarrassingly simple. Sample a clean image, pick a random $t$, sample a noise tensor $\epsilon$, build $x_t$ via the formula above, and ask a network to predict $\epsilon$ from $(x_t, t)$. The loss is plain MSE: $\| \epsilon_\theta(x_t, t) - \epsilon \|^2$. No adversarial game, no variational lower bound to optimize directly — just regression. At inference we run the *reverse process* iteratively: start from $x_T \sim \mathcal{N}(0, I)$ and walk back from $T$ to $0$, using the network's noise predictions to subtract noise step by step.

## Implementation

### Step 1 — Linear noise schedule

The schedule defines the per-step noise variance $\beta_t$. We precompute $\alpha_t$, $\bar\alpha_t$, and the two square-root terms used by the forward formula. Precomputing matters: every training step indexes into these tensors, so doing it once at construction avoids recomputing $T$ products on every batch.

In [ ]:
def linear_beta_schedule(T=200, beta_start=1e-4, beta_end=0.02):
    # T=200 is small for DDPM (paper used 1000) but enough to teach the dynamics.
    # Smaller T = faster sampling for the notebook, slightly cruder samples.
    betas = torch.linspace(beta_start, beta_end, T)
    # alpha_t = 1 - beta_t is the *retention* factor at step t.
    alphas = 1.0 - betas
    # alpha_bar_t is the cumulative retention from x_0 all the way to x_t.
    # This is what shows up in the closed-form forward formula.
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    # Pre-square-root these because every q_sample call needs them and sqrt is not free.
    sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
    sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
    return {
        'betas': betas, 'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'sqrt_alphas_cumprod': sqrt_alphas_cumprod,
        'sqrt_one_minus_alphas_cumprod': sqrt_one_minus_alphas_cumprod,
    }

T = 200
sched = {k: v.to(device) for k, v in linear_beta_schedule(T).items()}
# Sanity: alpha_bar starts near 1 (no noise) and ends near 0 (almost pure noise).
print(f'alpha_bar[0]   = {sched["alphas_cumprod"][0].item():.4f}  (should be ~1.0 — barely noisy)')
print(f'alpha_bar[T-1] = {sched["alphas_cumprod"][-1].item():.4f}  (should be ~0    — almost pure noise)')

### Step 2 — Closed-form forward sample $q(x_t | x_0)$

Given a clean image and a timestep, return a noisy image. We also return the noise we used, because the training target is exactly that noise.

In [ ]:
def q_sample(x_0, t, sched, noise=None):
    # t is a (batch,) tensor of integer timesteps. We gather the schedule values
    # at those timesteps and reshape so they broadcast against x_0 (B, C, H, W).
    if noise is None:
        noise = torch.randn_like(x_0)
    # gather lets each example in the batch use a different timestep — essential because
    # we sample t uniformly per example during training.
    sqrt_ab = sched['sqrt_alphas_cumprod'].gather(0, t).view(-1, 1, 1, 1)
    sqrt_one_minus_ab = sched['sqrt_one_minus_alphas_cumprod'].gather(0, t).view(-1, 1, 1, 1)
    return sqrt_ab * x_0 + sqrt_one_minus_ab * noise, noise

Let's verify that q_sample produces sensible images at a few timesteps. We'll grab a real MNIST digit and corrupt it at evenly-spaced t values. By $t=T-1$ it should look indistinguishable from random noise.

In [ ]:
# Normalize MNIST pixel values to [-1, 1] so they match the variance the noise will have.
# The forward formula assumes x_0 is roughly mean-0 unit-variance — uint8 [0, 255] would break the schedule.
mnist_transform = transforms.Compose([
    transforms.ToTensor(),                # -> [0, 1]
    transforms.Normalize((0.5,), (0.5,)),  # -> [-1, 1]
])
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=mnist_transform)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=mnist_transform)
print(f'train: {len(train_ds)}  test: {len(test_ds)}')
x_0_demo = train_ds[0][0].unsqueeze(0).to(device)  # (1, 1, 28, 28)
print(f'x_0 shape: {tuple(x_0_demo.shape)}  range: [{x_0_demo.min():.2f}, {x_0_demo.max():.2f}]')

In [ ]:
demo_ts = torch.tensor([0, T // 8, T // 4, T // 2, 3 * T // 4, T - 1], device=device)
fig, axes = plt.subplots(1, len(demo_ts), figsize=(2 * len(demo_ts), 2.2))
for ax, t in zip(axes, demo_ts):
    x_t, _ = q_sample(x_0_demo, t.unsqueeze(0), sched)
    ax.imshow(x_t.squeeze().cpu(), cmap='gray', vmin=-2, vmax=2)
    ax.set_title(f't={t.item()}'); ax.axis('off')
plt.suptitle('forward process: clean -> noise', y=1.05)
plt.tight_layout(); plt.show()

### Step 3 — Sinusoidal timestep embedding

The UNet must know *which* noise level it is denoising. We map an integer timestep into a continuous vector with sinusoidal features at multiple frequencies, then project that vector through a small MLP to feed the conv blocks. This is the same construction used by transformer positional encoding and by the original DDPM.

In [ ]:
def sinusoidal_embedding(t, dim):
    # t: (batch,) integer timesteps. dim: embedding size, must be even.
    # Returns (batch, dim) where odd indices use cos, even use sin, at log-spaced frequencies.
    # The log-spacing is what gives the embedding scale-invariance across t magnitudes.
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
    args = t[:, None].float() * freqs[None, :]  # (batch, half)
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

# Sanity check the shape and that the embedding actually varies with t.
te = sinusoidal_embedding(torch.arange(0, 5, device=device), dim=16)
print(f'embedding shape: {tuple(te.shape)}')
print(f'std across t (should be > 0): {te.std(dim=0).mean().item():.3f}')

### Step 4 — Tiny UNet with FiLM-style timestep conditioning

A real DDPM UNet has residual blocks, attention at low resolution, and several downsample/upsample stages. We strip that down to two scales (28→14→7) with two conv-norm-SiLU blocks per scale. The timestep embedding is mixed in by projecting it to channel-width and *adding* it to each block's output (FiLM-style additive bias). Total params under 1M so training fits in 5-10 minutes on a T4.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        # GroupNorm because BatchNorm interacts badly with the per-example random t in DDPM.
        # GroupNorm normalizes per-sample, so different timesteps in a batch don't pollute each other.
        self.norm1 = nn.GroupNorm(8, in_ch) if in_ch >= 8 else nn.GroupNorm(1, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        # Project the timestep embedding to out_ch and broadcast it as an additive bias.
        # FiLM-style: same value added to every spatial location, but different per channel.
        self.t_proj = nn.Linear(t_dim, out_ch)
        # 1x1 residual to match channels when in_ch != out_ch.
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        # Add timestep bias *between* the two convs. (B, out_ch, 1, 1) broadcasts over H, W.
        h = h + self.t_proj(t_emb)[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)

In [ ]:
class TinyUNet(nn.Module):
    def __init__(self, in_ch=1, base=32, t_dim=64):
        super().__init__()
        self.t_dim = t_dim
        # Sinusoidal -> MLP. The MLP gives the timestep embedding learnable shape.
        self.t_mlp = nn.Sequential(
            nn.Linear(t_dim, t_dim * 4),
            nn.SiLU(),
            nn.Linear(t_dim * 4, t_dim * 4),
        )
        td = t_dim * 4  # post-MLP timestep width
        # Down path: 28 -> 14 -> 7
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.d1 = ConvBlock(base, base, td)
        self.down1 = nn.Conv2d(base, base * 2, 4, stride=2, padding=1)   # 28->14
        self.d2 = ConvBlock(base * 2, base * 2, td)
        self.down2 = nn.Conv2d(base * 2, base * 4, 4, stride=2, padding=1)  # 14->7
        # Bottleneck at 7x7
        self.mid = ConvBlock(base * 4, base * 4, td)
        # Up path with skip connections from the down path (concatenated, then channel-reduced)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 4, stride=2, padding=1)  # 7->14
        self.u2 = ConvBlock(base * 4, base * 2, td)   # input is up2 + skip from d2 (cat doubles channels)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 4, stride=2, padding=1)      # 14->28
        self.u1 = ConvBlock(base * 2, base, td)       # input is up1 + skip from d1
        self.out_conv = nn.Conv2d(base, in_ch, 3, padding=1)
    def forward(self, x, t):
        # Build the timestep embedding once and reuse across blocks.
        te = sinusoidal_embedding(t, self.t_dim)
        te = self.t_mlp(te)
        x0 = self.in_conv(x)
        d1 = self.d1(x0, te)            # (B, base, 28, 28)
        d2 = self.d2(self.down1(d1), te)  # (B, base*2, 14, 14)
        m  = self.mid(self.down2(d2), te) # (B, base*4, 7, 7)
        # Concatenate skip features along the channel axis — classic UNet shape.
        u2 = self.u2(torch.cat([self.up2(m), d2], dim=1), te)
        u1 = self.u1(torch.cat([self.up1(u2), d1], dim=1), te)
        return self.out_conv(u1)

model = TinyUNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'TinyUNet params: {n_params:,}  ({n_params / 1e6:.2f} M)')
# Quick forward-shape check before we wire up training.
x_dummy = torch.randn(4, 1, 28, 28, device=device)
t_dummy = torch.randint(0, T, (4,), device=device)
with torch.no_grad():
    out = model(x_dummy, t_dummy)
print(f'forward shape: {tuple(out.shape)}  range: [{out.min().item():.2f}, {out.max().item():.2f}]')

### Step 5 — One training step

Pick a random timestep per example, build the noisy image with `q_sample`, predict noise, MSE against the true noise. That is the entire training objective.

In [ ]:
def ddpm_train_step(model, x_0, sched, T):
    # Uniform t per example. The lesson explains why uniform: every noise level needs equal training signal.
    t = torch.randint(0, T, (x_0.size(0),), device=x_0.device)
    x_t, noise = q_sample(x_0, t, sched)
    pred_noise = model(x_t, t)
    return F.mse_loss(pred_noise, noise)

## Experiment — Train DDPM on MNIST

In [ ]:
BATCH = 128
EPOCHS = 8 if device == 'cuda' else 2
loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
opt = torch.optim.Adam(model.parameters(), lr=2e-4)
losses = []
log_every = 100
step = 0
t0 = time.time()
for epoch in range(EPOCHS):
    for x_0, _ in loader:
        x_0 = x_0.to(device, non_blocking=True)
        loss = ddpm_train_step(model, x_0, sched, T)
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % log_every == 0:
            elapsed = time.time() - t0
            print(f'epoch {epoch}  step {step:5d}  loss {loss.item():.4f}  ({elapsed:.0f}s elapsed)')
        step += 1
print(f'done. total {time.time() - t0:.0f}s, final loss {losses[-1]:.4f}')

In [ ]:
# Smooth the loss with a moving average — raw per-step loss bounces because t is sampled uniformly
# and predicting noise at large t is intrinsically easier than at small t (the target IS approximately the input).
import numpy as np
window = 50
smoothed = np.convolve(losses, np.ones(window) / window, mode='valid')
plt.figure(figsize=(7, 3.5))
plt.plot(losses, alpha=0.25, label='raw')
plt.plot(np.arange(window - 1, len(losses)), smoothed, label=f'{window}-step MA')
plt.xlabel('step'); plt.ylabel('MSE noise loss')
plt.title('DDPM training loss')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## Sampling — DDPM reverse process

Reverse sampling is the iterative loop we couldn't shortcut. At each step $t$ we predict noise, subtract a scaled version, and add a little fresh noise (the stochastic term that distinguishes DDPM from DDIM). We stash intermediate $x_t$ values at evenly-spaced indices so we can later plot the denoising trajectory.

In [ ]:
@torch.no_grad()
def p_sample_loop(model, sched, T, shape, record_intermediate=True, n_intermediate=8):
    # Start from pure Gaussian noise — this is x_T.
    x = torch.randn(shape, device=device)
    # Pick which timesteps to remember for visualization.
    record_at = set(torch.linspace(T - 1, 0, n_intermediate).round().long().tolist()) if record_intermediate else set()
    snaps = {}
    betas = sched['betas']
    alphas = sched['alphas']
    alphas_cumprod = sched['alphas_cumprod']
    sqrt_one_minus_ab = sched['sqrt_one_minus_alphas_cumprod']
    for t in reversed(range(T)):
        # Build a (batch,) timestep tensor — every example in the batch is at the same t during sampling.
        t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
        eps = model(x, t_batch)
        # Posterior mean: see DDPM eq. 11. Subtract a scaled noise prediction from x_t.
        mean = (1.0 / alphas[t].sqrt()) * (x - (betas[t] / sqrt_one_minus_ab[t]) * eps)
        if t > 0:
            # Stochastic term — sigma_t is set to sqrt(beta_t) (the simpler of the two DDPM choices).
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            # At t=0 we don't add fresh noise; the mean *is* the final x_0 estimate.
            x = mean
        if t in record_at:
            snaps[t] = x.detach().cpu().clone()
    return x, snaps

In [ ]:
model.eval()
N_TEST_SAMPLES = 8 if device == 'cuda' else 4
t0 = time.time()
samples, snaps = p_sample_loop(model, sched, T, (N_TEST_SAMPLES, 1, 28, 28))
print(f'DDPM sampling ({T} steps, {N_TEST_SAMPLES} images): {time.time() - t0:.1f}s')
print(f'sample range: [{samples.min().item():.2f}, {samples.max().item():.2f}]')

In [ ]:
# Plot the denoising trajectory: rows = sample index, columns = decreasing t (noise -> digit).
ts_sorted = sorted(snaps.keys(), reverse=True)
fig, axes = plt.subplots(N_TEST_SAMPLES, len(ts_sorted), figsize=(1.4 * len(ts_sorted), 1.4 * N_TEST_SAMPLES))
for r in range(N_TEST_SAMPLES):
    for c, t in enumerate(ts_sorted):
        ax = axes[r, c] if N_TEST_SAMPLES > 1 else axes[c]
        ax.imshow(snaps[t][r, 0], cmap='gray', vmin=-2, vmax=2)
        ax.axis('off')
        if r == 0:
            ax.set_title(f't={t}', fontsize=9)
plt.suptitle('reverse process: noise -> digit  (left=high noise, right=clean)', y=1.0)
plt.tight_layout(); plt.show()

After the early columns the model has nothing to work with — it is staring into white noise. Around $t \approx T/2$ a vague structure emerges, and over the last quarter of timesteps the digit sharpens into something recognizable. If your samples look like noise-blobs, train for more epochs; if they look like blurry noise even at $t=0$, your model is undertrained and predicting near-zero noise (which makes the reverse process collapse to identity).

## Variations

### DDIM: 25-step deterministic sampling

DDIM (Song et al. 2020) reuses the *same trained network* but replaces the stochastic reverse step with a deterministic ODE-style update, and skips most timesteps. With 25 steps instead of 200 we expect roughly 8x faster sampling at slightly degraded quality.

In [ ]:
@torch.no_grad()
def ddim_sample(model, sched, T, shape, n_steps=25):
    # Choose a subset of timesteps from T-1 down to 0. Linearly spaced is the simplest choice;
    # quadratic spacing (more steps near 0) often gives slightly better samples.
    step_indices = torch.linspace(T - 1, 0, n_steps + 1).round().long()
    x = torch.randn(shape, device=device)
    alphas_cumprod = sched['alphas_cumprod']
    for i in range(n_steps):
        t = step_indices[i].item()
        t_next = step_indices[i + 1].item()
        ab_t = alphas_cumprod[t]
        ab_next = alphas_cumprod[t_next]
        t_batch = torch.full((shape[0],), t, device=device, dtype=torch.long)
        eps = model(x, t_batch)
        # Predict x_0 from (x_t, eps) using the closed-form forward formula, solved for x_0.
        x0_pred = (x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()
        # Re-noise to the next (less noisy) timestep using the *same* eps direction.
        # eta=0 deterministic case: no fresh randomness, hence ~8x fewer steps still works.
        x = ab_next.sqrt() * x0_pred + (1 - ab_next).sqrt() * eps
    return x

t0 = time.time()
ddim_samples = ddim_sample(model, sched, T, (N_TEST_SAMPLES, 1, 28, 28), n_steps=25)
ddim_time = time.time() - t0
t0 = time.time()
ddpm_samples, _ = p_sample_loop(model, sched, T, (N_TEST_SAMPLES, 1, 28, 28), record_intermediate=False)
ddpm_time = time.time() - t0
print(f'DDPM (200 steps): {ddpm_time:.2f}s')
print(f'DDIM ( 25 steps): {ddim_time:.2f}s   speedup={ddpm_time/ddim_time:.1f}x')

In [ ]:
fig, axes = plt.subplots(2, N_TEST_SAMPLES, figsize=(1.4 * N_TEST_SAMPLES, 3))
for c in range(N_TEST_SAMPLES):
    axes[0, c].imshow(ddpm_samples[c, 0].cpu(), cmap='gray', vmin=-2, vmax=2); axes[0, c].axis('off')
    axes[1, c].imshow(ddim_samples[c, 0].cpu(), cmap='gray', vmin=-2, vmax=2); axes[1, c].axis('off')
axes[0, 0].set_ylabel('DDPM 200', fontsize=10)
axes[1, 0].set_ylabel('DDIM 25', fontsize=10)
plt.suptitle('DDPM vs DDIM with the same trained network')
plt.tight_layout(); plt.show()

### Cosine schedule

Nichol & Dhariwal (2021) observed that the linear schedule destroys signal too quickly at the start — the early-step betas are already big enough that there is barely anything to denoise. The cosine schedule keeps $\bar\alpha_t$ near 1 for longer at the start and drops smoothly to 0 at the end, spending more compute on the useful middle range. We won't retrain the full model — the comparison here is the *schedules themselves*.

In [ ]:
def cosine_beta_schedule(T=200, s=0.008):
    # Following Nichol & Dhariwal: define alpha_bar via cos^2 and recover beta_t from it.
    steps = torch.arange(T + 1, dtype=torch.float64) / T
    f = torch.cos(((steps + s) / (1 + s)) * math.pi / 2) ** 2
    alphas_cumprod = (f / f[0]).float()
    # beta_t = 1 - alpha_bar_t / alpha_bar_{t-1}, clamped to keep numerical sanity.
    betas = 1 - alphas_cumprod[1:] / alphas_cumprod[:-1]
    betas = torch.clamp(betas, 1e-5, 0.999)
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    return {
        'betas': betas, 'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'sqrt_alphas_cumprod': torch.sqrt(alphas_cumprod),
        'sqrt_one_minus_alphas_cumprod': torch.sqrt(1.0 - alphas_cumprod),
    }

lin = linear_beta_schedule(T)
cos_ = cosine_beta_schedule(T)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(lin['betas'], label='linear'); ax[0].plot(cos_['betas'], label='cosine')
ax[0].set_title(r'$\beta_t$ per step'); ax[0].set_xlabel('t'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(lin['alphas_cumprod'], label='linear'); ax[1].plot(cos_['alphas_cumprod'], label='cosine')
ax[1].set_title(r'$\bar\alpha_t$ (signal retention)'); ax[1].set_xlabel('t'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('Notice the cosine schedule keeps alpha_bar near 1 longer — more useful timesteps before signal is gone.')

In [ ]:
# Sanity check: with an *untrained* fresh UNet, just compare what q_sample produces under the two schedules.
# This won't show sample quality (we'd need to retrain), but it shows the corruption profile difference.
x_demo = train_ds[0][0].unsqueeze(0).to(device)
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for j, t_val in enumerate([0, T // 8, T // 4, T // 2, 3 * T // 4, T - 1]):
    sched_lin = {k: v.to(device) for k, v in lin.items()}
    sched_cos = {k: v.to(device) for k, v in cos_.items()}
    t_b = torch.tensor([t_val], device=device)
    xL, _ = q_sample(x_demo, t_b, sched_lin)
    xC, _ = q_sample(x_demo, t_b, sched_cos)
    axes[0, j].imshow(xL.squeeze().cpu(), cmap='gray', vmin=-2, vmax=2); axes[0, j].axis('off')
    axes[1, j].imshow(xC.squeeze().cpu(), cmap='gray', vmin=-2, vmax=2); axes[1, j].axis('off')
    axes[0, j].set_title(f't={t_val}', fontsize=9)
axes[0, 0].set_ylabel('linear', fontsize=10)
axes[1, 0].set_ylabel('cosine', fontsize=10)
plt.suptitle('cosine retains structure longer at intermediate t')
plt.tight_layout(); plt.show()

## What to try next

- **Class conditioning + classifier-free guidance.** Pass the digit label through an embedding, add it to `t_emb`, and during training drop the label 10% of the time (replace with a learned null token). At sampling time, run two forward passes (with and without the label) and combine: $\hat\epsilon = \epsilon_u + w\,(\epsilon_c - \epsilon_u)$. Watch how cranking $w$ from 0 to 5 sharpens the digit class but introduces artifacts.
- **Latent diffusion.** Train the VAE from the previous notebook on MNIST, freeze it, then run this same DDPM on the *latents* instead of the pixels. You will discover that diffusion in a 4-channel 7x7 latent space is roughly 16x faster than diffusion in a 1-channel 28x28 pixel space — this is the same trick Stable Diffusion uses at scale.
- **Score-matching framing.** The noise prediction $\epsilon_\theta$ is equivalent (up to scaling) to learning $\nabla_x \log p_t(x)$ — the score of the noisy data distribution. Re-derive `p_sample_loop` using Langevin dynamics on the score and verify the samples match. Useful prep for reading the Song et al. score-SDE papers.
- **Fewer parameters, more depth.** Halve `base` channels and add a third downsampling stage. Compare params, training time, and sample quality. Diffusion benefits more from depth than from width per parameter, an effect that gets stronger as resolution increases.